In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from skimage.io import imread
from skimage.color import gray2rgb
from skimage.morphology import (
    remove_small_holes,
    remove_small_objects,
    binary_opening,
    binary_closing,
    disk,
)
from skimage.measure import label, regionprops
from skimage.segmentation import clear_border
from skimage.exposure import equalize_adapthist
from skimage.restoration import denoise_bilateral

import matplotlib.pyplot as plt

np.random.seed(100)



In [2]:
try:
    import pydensecrf.densecrf as dcrf  # noqa: F401
    from pydensecrf.utils import unary_from_labels  # noqa: F401

    PYDENSECRF_AVAILABLE = True
except ModuleNotFoundError:
    PYDENSECRF_AVAILABLE = False

print("PYDENSECRF_AVAILABLE:", PYDENSECRF_AVAILABLE)




PYDENSECRF_AVAILABLE: False


In [3]:
def rle_decode(rle_mask, shape=(101, 101)):
    """
    rle_mask: run-length as string formatted (start length)
    Returns numpy array, 1 - mask, 0 - background, shape (101, 101)

    NOTE: Competition expects pixels numbered top-to-bottom then left-to-right,
    which corresponds to Fortran order when flattening a (H, W) array.
    """
    if rle_mask is None:
        return np.zeros(shape, dtype=np.uint8)
    if isinstance(rle_mask, float) and np.isnan(rle_mask):
        return np.zeros(shape, dtype=np.uint8)
    rle_mask = str(rle_mask).strip()
    if rle_mask == "" or rle_mask.lower() == "nan":
        return np.zeros(shape, dtype=np.uint8)

    s = rle_mask.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")


def rle_encode(im):
    """
    im: numpy array, 1 - mask, 0 - background
    Returns run length as string formatted.

    NOTE: Must use Fortran order to match Kaggle TGS salt RLE convention.
    """
    im = (im > 0).astype(np.uint8)
    pixels = im.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


def _assert_rle_convention_ok():
    m = np.zeros((101, 101), dtype=np.uint8)
    m[0, 0] = 1
    m[100, 100] = 1
    m[50:55, 10:12] = 1
    r = rle_encode(m)
    m2 = rle_decode(r, shape=m.shape)
    if not np.array_equal(m, m2):
        raise RuntimeError(
            "RLE encode/decode convention mismatch (likely wrong flatten order)."
        )


_assert_rle_convention_ok()
print("RLE convention sanity check: OK")



RLE convention sanity check: OK


In [4]:
BASE = "../input/tgs-salt-identification-challenge"
test_path = os.path.join(BASE, "test", "images")
sample_path = os.path.join(BASE, "sample_submission.csv")
train_csv_path = os.path.join(BASE, "train.csv")
depths_path = os.path.join(BASE, "depths.csv")
train_masks_dir = os.path.join(BASE, "train", "masks")
train_images_dir = os.path.join(BASE, "train", "images")

assert os.path.exists(sample_path), f"Missing sample_submission at {sample_path}"
assert os.path.isdir(test_path), f"Missing test images dir at {test_path}"
assert os.path.exists(train_csv_path), f"Missing train.csv at {train_csv_path}"
assert os.path.exists(depths_path), f"Missing depths.csv at {depths_path}"
assert os.path.isdir(train_masks_dir), f"Missing train masks dir at {train_masks_dir}"
assert os.path.isdir(
    train_images_dir
), f"Missing train images dir at {train_images_dir}"

df = pd.read_csv(sample_path)
if "id" not in df.columns or "rle_mask" not in df.columns:
    raise ValueError(f"Unexpected submission columns: {df.columns.tolist()}")

print("Loaded sample_submission:", df.shape)
print(df.head())



Loaded sample_submission: (1000, 2)
           id rle_mask
0  003c477d7c      1 1
1  0108518d1e      1 1
2  010ee525b6      1 1
3  01323211a0      1 1
4  026b509cd4      1 1


In [5]:
train_df = pd.read_csv(train_csv_path)
if "id" not in train_df.columns:
    raise ValueError(f"Unexpected train.csv columns: {train_df.columns.tolist()}")

depths_df = pd.read_csv(depths_path)
if "id" not in depths_df.columns or "z" not in depths_df.columns:
    raise ValueError(f"Unexpected depths.csv columns: {depths_df.columns.tolist()}")

train_merged = train_df[["id"]].merge(depths_df, on="id", how="left")
if train_merged["z"].isna().any():
    raise ValueError(
        "Some train ids are missing depths; cannot depth-calibrate safely."
    )

z_train = train_merged["z"].values.astype(np.float32)

pos_fracs = np.zeros(len(train_merged), dtype=np.float32)
is_empty = np.zeros(len(train_merged), dtype=np.float32)
missing_masks = 0
for i, image_id in enumerate(
    tqdm(
        train_merged["id"].values,
        desc="Compute train positive fractions from mask PNGs",
    )
):
    mpath = os.path.join(train_masks_dir, f"{image_id}.png")
    if not os.path.exists(mpath):
        missing_masks += 1
        pos_fracs[i] = 0.0
        is_empty[i] = 1.0
        continue
    m = imread(mpath)
    if m.ndim == 3:
        m = m[:, :, 0]
    m = (m > 127).astype(np.uint8)
    frac = float(m.mean())
    pos_fracs[i] = frac
    is_empty[i] = 1.0 if frac <= 0.0 else 0.0

if missing_masks > 0:
    print(f"Warning: missing {missing_masks} train mask PNGs; treated as empty.")

GLOBAL_POS_FRAC = float(np.clip(pos_fracs.mean(), 1e-4, 0.9999))
GLOBAL_EMPTY_RATE = float(np.clip(is_empty.mean(), 0.0, 1.0))
print("Estimated GLOBAL_POS_FRAC from train masks:", GLOBAL_POS_FRAC)
print("Estimated GLOBAL_EMPTY_RATE from train masks:", GLOBAL_EMPTY_RATE)

n_bins = 20
bin_edges = np.quantile(z_train, np.linspace(0, 1, n_bins + 1))
bin_edges = np.unique(bin_edges)
if len(bin_edges) < 3:
    bin_edges = np.array([z_train.min(), z_train.max()], dtype=np.float32)

train_bins = np.digitize(z_train, bin_edges[1:-1], right=True)

bin_median_frac = np.full(int(train_bins.max()) + 1, np.nan, dtype=np.float32)
bin_empty_rate = np.full(int(train_bins.max()) + 1, np.nan, dtype=np.float32)
for b in np.unique(train_bins):
    mask = train_bins == b
    if mask.sum() == 0:
        continue
    bin_median_frac[int(b)] = float(np.median(pos_fracs[mask]))
    bin_empty_rate[int(b)] = float(np.mean(is_empty[mask]))

for arr, fallback in [
    (bin_median_frac, GLOBAL_POS_FRAC),
    (bin_empty_rate, GLOBAL_EMPTY_RATE),
]:
    nan_mask = np.isnan(arr)
    arr[nan_mask] = fallback


def _smooth_1d(x, k=3):
    if k <= 1:
        return x.copy()
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(k, dtype=np.float32) / float(k)
    return np.convolve(xp, kernel, mode="valid").astype(np.float32)


bin_median_frac_sm = _smooth_1d(bin_median_frac.astype(np.float32), k=3)
bin_empty_rate_sm = _smooth_1d(bin_empty_rate.astype(np.float32), k=3)

print(
    "Depth bins:", len(bin_edges) - 1, "bins array size:", bin_median_frac_sm.shape[0]
)


def _bin_index_from_depth(z_value):
    return int(
        np.digitize(
            np.array([z_value], dtype=np.float32),
            bin_edges[1:-1],
            right=True,
        )[0]
    )


def expected_pos_frac_from_depth(z_value):
    b = _bin_index_from_depth(z_value)
    b = int(np.clip(b, 0, len(bin_median_frac_sm) - 1))
    return float(np.clip(bin_median_frac_sm[b], 5e-4, 0.35))


def expected_empty_rate_from_depth(z_value):
    b = _bin_index_from_depth(z_value)
    b = int(np.clip(b, 0, len(bin_empty_rate_sm) - 1))
    return float(np.clip(bin_empty_rate_sm[b], 0.0, 1.0))


def pseudo_depth_from_image(img):
    img = img.astype(np.float32)
    if img.ndim == 3:
        img = img[:, :, 0]
    mn, mx = float(img.min()), float(img.max())
    if mx > mn:
        x = (img - mn) / (mx - mn)
    else:
        x = img * 0.0
    z_min, z_max = float(z_train.min()), float(z_train.max())
    return z_min + float(x.mean()) * (z_max - z_min)




Compute train positive fractions from mask PNGs: 100%|██████████| 3000/3000 [00:01<00:00, 1801.30it/s]

Estimated GLOBAL_POS_FRAC from train masks: 0.24872376024723053
Estimated GLOBAL_EMPTY_RATE from train masks: 0.38999998569488525
Depth bins: 20 bins array size: 20


In [6]:
def crf(original_image, mask_img):
    """
    If pydensecrf is available, use DenseCRF refinement (original intent).
    Otherwise, apply a deterministic morphology-based refinement that keeps
    the decode->refine->encode semantics.
    """
    if mask_img is None:
        mask_img = np.zeros((101, 101), dtype=np.uint8)
    if mask_img.ndim == 3:
        mask2d = mask_img[:, :, 0]
    else:
        mask2d = mask_img

    mask2d = (mask2d > 0).astype(np.uint8)

    if PYDENSECRF_AVAILABLE:
        if len(mask_img.shape) < 3:
            mask_img_rgb = gray2rgb(mask2d)
        else:
            mask_img_rgb = mask_img

        annotated_label = (
            mask_img_rgb[:, :, 0]
            + (mask_img_rgb[:, :, 1] << 8)
            + (mask_img_rgb[:, :, 2] << 16)
        )
        _, labels = np.unique(annotated_label, return_inverse=True)

        n_labels = 2
        d = dcrf.DenseCRF2D(original_image.shape[1], original_image.shape[0], n_labels)

        U = unary_from_labels(labels, n_labels, gt_prob=0.7, zero_unsure=False)
        d.setUnaryEnergy(U)

        d.addPairwiseGaussian(
            sxy=(3, 3),
            compat=3,
            kernel=dcrf.DIAG_KERNEL,
            normalization=dcrf.NORMALIZE_SYMMETRIC,
        )

        Q = d.inference(10)
        MAP = np.argmax(Q, axis=0)
        return MAP.reshape((original_image.shape[0], original_image.shape[1])).astype(
            np.uint8
        )

    m = mask2d.astype(bool)
    se = disk(1)
    m = binary_opening(m, se)
    m = binary_closing(m, se)
    m = remove_small_objects(m, min_size=15)
    m = remove_small_holes(m, area_threshold=15)
    return m.astype(np.uint8)




In [7]:
def _normalize01(img):
    img = img.astype(np.float32)
    if img.ndim == 3:
        img = img[:, :, 0]
    mn, mx = float(img.min()), float(img.max())
    if mx > mn:
        return (img - mn) / (mx - mn)
    return img * 0.0


def _mask_fragmentation(m_u8):
    m = (m_u8 > 0).astype(np.uint8)
    area = int(m.sum())
    if area == 0:
        return 0.0
    cc = label(m, connectivity=1)
    n_cc = int(cc.max())
    return float(n_cc) / float(area)


def _mask_area_penalty(m_u8, expected_pos_frac):
    m = (m_u8 > 0).astype(np.uint8)
    frac = float(m.mean())
    return float(abs(frac - float(expected_pos_frac)))


def _scalar_field_for_thresholding(img):
    """
    Deterministic region cue (kept as-is): CLAHE + light denoise.
    """
    x = _normalize01(img)
    x = equalize_adapthist(x, clip_limit=0.01).astype(np.float32)
    x = denoise_bilateral(
        x, sigma_color=0.08, sigma_spatial=2, channel_axis=None
    ).astype(np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx > mn:
        x = (x - mn) / (mx - mn)
    else:
        x = x * 0.0
    return x


def _iou_binary(pred_u8, true_u8):
    # Change (score-relevant, minimal): use IoU as calibration objective, closer to Kaggle metric.
    p = pred_u8.astype(bool)
    t = true_u8.astype(bool)
    inter = np.logical_and(p, t).sum()
    union = np.logical_or(p, t).sum()
    if union == 0:
        return 1.0
    return float(inter) / float(union)


def _calibrate_thresholds_from_train(
    train_ids,
    train_bins,
    max_per_bin=80,
    seed=100,
):
    """
    Change (score-relevant, minimal): calibrate per-bin polarity+threshold by maximizing mean IoU
    on a small bin sample (using the SAME scalar field + simple thresholding as inference).
    This preserves the pipeline and tends to align much better with MAP@IoU than pure area matching.
    """
    rng = np.random.RandomState(seed)
    n_bins_local = int(train_bins.max()) + 1

    thr_grid = np.concatenate(
        [
            np.linspace(0.05, 0.95, 19, dtype=np.float32),
            np.linspace(0.02, 0.98, 25, dtype=np.float32),
        ]
    )
    thr_grid = np.unique(np.clip(thr_grid, 0.0, 1.0))

    bin_polarity = np.full(n_bins_local, 1, dtype=np.int8)
    bin_thr = np.full(n_bins_local, np.float32(0.5), dtype=np.float32)
    bin_sep = np.zeros(n_bins_local, dtype=np.float32)

    for b in range(n_bins_local):
        idx = np.where(train_bins == b)[0]
        if idx.size == 0:
            continue
        if idx.size > max_per_bin:
            idx = rng.choice(idx, size=max_per_bin, replace=False)

        xs = []
        ys = []
        for i in idx:
            image_id = str(train_ids[i])
            ipath = os.path.join(train_images_dir, f"{image_id}.png")
            mpath = os.path.join(train_masks_dir, f"{image_id}.png")
            if not (os.path.exists(ipath) and os.path.exists(mpath)):
                continue

            img = imread(ipath)
            m = imread(mpath)
            if m.ndim == 3:
                m = m[:, :, 0]
            y = (m > 127).astype(np.uint8)

            x = _scalar_field_for_thresholding(img).astype(np.float32)
            xs.append(x)
            ys.append(y)

        if len(xs) < 8:
            continue

        def eval_pol(pol):
            scores = []
            for thr in thr_grid:
                s = 0.0
                for x, y in zip(xs, ys):
                    if pol >= 0:
                        pred = (x > thr).astype(np.uint8)
                    else:
                        pred = (x < thr).astype(np.uint8)
                    s += _iou_binary(pred, y)
                scores.append(s / float(len(xs)))
            scores = np.asarray(scores, dtype=np.float32)
            j_best = int(np.argmax(scores))
            return (
                float(thr_grid[j_best]),
                float(scores[j_best]),
                float(scores.max() - scores.min()),
            )

        thr_p, score_p, spread_p = eval_pol(+1)
        thr_n, score_n, spread_n = eval_pol(-1)

        if score_n > score_p:
            bin_polarity[b] = -1
            bin_thr[b] = np.float32(thr_n)
            bin_sep[b] = np.float32(spread_n)
        else:
            bin_polarity[b] = 1
            bin_thr[b] = np.float32(thr_p)
            bin_sep[b] = np.float32(spread_p)

    valid = ~np.isnan(bin_thr)
    if valid.any():
        global_thr = float(np.median(bin_thr[valid]))
        global_pol = int(np.round(np.median(bin_polarity[valid])).astype(int))
        global_pol = 1 if global_pol >= 0 else -1
    else:
        global_thr, global_pol = 0.5, 1

    bin_thr[np.isnan(bin_thr)] = np.float32(global_thr)

    return bin_polarity, bin_thr, bin_sep


bin_polarity, bin_thr, bin_sep = _calibrate_thresholds_from_train(
    train_ids=train_merged["id"].values,
    train_bins=train_bins,
    max_per_bin=80,
    seed=100,
)
print(
    "Calibrated per-bin thresholds. Example (bin 0):",
    int(bin_polarity[0]),
    float(bin_thr[0]),
    float(bin_sep[0]),
)


def predict_mask_from_image(img, expected_pos_frac=GLOBAL_POS_FRAC, z_value=None):
    """
    Core logic preserved: scalar field -> threshold -> morphology refine downstream.
    Change (score-relevant, minimal): trust IoU-calibrated threshold more often when it shows
    non-trivial separation; keep existing quantile-area fallback unchanged.
    """
    x = _scalar_field_for_thresholding(img)

    if z_value is None or (isinstance(z_value, float) and np.isnan(z_value)):
        b = None
    else:
        b = _bin_index_from_depth(float(z_value))
        b = int(np.clip(b, 0, len(bin_thr) - 1))

    use_calibrated = False
    if b is not None:
        # Change: slightly lower confidence threshold since IoU-spread is usually smaller than area-error spread,
        # improving usage of the better-calibrated per-bin rule.
        if float(bin_sep[b]) >= 0.006:
            use_calibrated = True

    if use_calibrated:
        thr = float(bin_thr[b])
        pol = int(bin_polarity[b])
        if pol >= 0:
            m = (x > thr).astype(np.uint8)
        else:
            m = (x < thr).astype(np.uint8)

        area_pen = _mask_area_penalty(m, expected_pos_frac)
        if area_pen <= 0.30:
            return m

    q_hi = float(np.clip(1.0 - expected_pos_frac, 0.0, 1.0))
    thr_hi = float(np.quantile(x, q_hi))
    m_pos = (x > thr_hi).astype(np.uint8)

    q_lo = float(np.clip(expected_pos_frac, 0.0, 1.0))
    thr_lo = float(np.quantile(x, q_lo))
    m_neg = (x < thr_lo).astype(np.uint8)

    frag_pos = _mask_fragmentation(m_pos)
    frag_neg = _mask_fragmentation(m_neg)
    area_pos = _mask_area_penalty(m_pos, expected_pos_frac)
    area_neg = _mask_area_penalty(m_neg, expected_pos_frac)

    score_pos = frag_pos + 0.25 * area_pos
    score_neg = frag_neg + 0.25 * area_neg

    if score_neg < score_pos:
        return m_neg
    if score_pos < score_neg:
        return m_pos
    if int(m_neg.sum()) < int(m_pos.sum()):
        return m_neg
    return m_pos


def _keep_largest_component_and_fill(
    mask_u8, min_area_keep=30, hole_area=30, only_if_fragmented=True
):
    m = (mask_u8 > 0).astype(np.uint8)
    area = int(m.sum())
    if area == 0:
        return m

    cc = label(m, connectivity=1)
    n = int(cc.max())
    if n <= 1:
        m2 = remove_small_holes(m.astype(bool), area_threshold=int(hole_area))
        return m2.astype(np.uint8)

    if only_if_fragmented:
        props = regionprops(cc)
        areas = np.array([p.area for p in props], dtype=np.float32)
        largest_ratio = float(areas.max() / max(1.0, areas.sum()))
        if largest_ratio >= 0.85:
            m2 = remove_small_holes(m.astype(bool), area_threshold=int(hole_area))
            return m2.astype(np.uint8)

    counts = np.bincount(cc.ravel().astype(np.int32))
    counts[0] = 0
    largest = int(np.argmax(counts))
    m_largest = cc == largest

    if int(m_largest.sum()) < int(min_area_keep):
        return np.zeros_like(m, dtype=np.uint8)

    m_largest = remove_small_holes(m_largest, area_threshold=int(hole_area))
    return m_largest.astype(np.uint8)


def _final_fill_step(mask_u8):
    m = mask_u8 > 0
    m = binary_closing(m, disk(1))
    m = remove_small_holes(m, area_threshold=40)
    return m.astype(np.uint8)




Calibrated per-bin thresholds. Example (bin 0): 1 0.30000001192092896 0.0525277778506279


In [8]:
test_depths = depths_df.merge(df[["id"]], on="id", how="right")
test_depth_map = dict(zip(test_depths["id"].values, test_depths["z"].values))

pred_rles = []
for image_id in tqdm(df["id"].values, desc="Predict+Refine"):
    img_path = os.path.join(test_path, f"{image_id}.png")
    img = imread(img_path)

    z_val = test_depth_map.get(image_id, np.nan)
    if z_val is None or (isinstance(z_val, float) and np.isnan(z_val)):
        z_val = float(pseudo_depth_from_image(img))
    else:
        z_val = float(z_val)

    exp_frac = expected_pos_frac_from_depth(z_val)
    exp_empty = expected_empty_rate_from_depth(z_val)

    raw_mask = predict_mask_from_image(img, expected_pos_frac=exp_frac, z_value=z_val)

    refined = crf(img, raw_mask)

    refined = clear_border(refined > 0).astype(np.uint8)

    do_largest_only = True
    if exp_frac >= 0.02:
        do_largest_only = False

    if do_largest_only:
        refined = _keep_largest_component_and_fill(
            refined, min_area_keep=30, hole_area=30, only_if_fragmented=True
        )
    else:
        m = refined > 0
        m = remove_small_objects(m, min_size=15)
        m = remove_small_holes(m, area_threshold=30)
        refined = m.astype(np.uint8)

    refined = _final_fill_step(refined)

    if (exp_frac <= 0.001 or exp_empty >= 0.95) and (int(refined.sum()) <= 5):
        refined = np.zeros((101, 101), dtype=np.uint8)

    rle = rle_encode(refined)
    if rle.strip() == "":
        rle = ""
    pred_rles.append(rle)

df["rle_mask"] = pred_rles



Predict+Refine: 100%|██████████| 1000/1000 [00:22<00:00, 44.61it/s]


In [9]:
assert df.shape[0] == 1000, "Expected 1000 test rows"
assert (df["id"].astype(str).str.len() > 0).all(), "Empty ids found"
assert list(df.columns) == [
    "id",
    "rle_mask",
], f"Column order mismatch: {df.columns.tolist()}"

out_path = "submission.csv"
df.to_csv(out_path, index=False)
print("Wrote submission:", out_path, "shape:", df.shape)
print(df.head())

Wrote submission: submission.csv shape: (1000, 2)
           id                                           rle_mask
0  003c477d7c  229 2 329 4 430 4 531 4 632 4 733 4 766 2 834 ...
1  0108518d1e                                                   
2  010ee525b6                                                   
3  01323211a0  2410 1 2510 3 2610 4 2710 5 2811 5 2912 5 3012...
4  026b509cd4  2008 1 2108 3 2210 3 2312 3 2414 2 2515 3 2615...
